# Conditional flow matching for TAL-NoiseMaker presets

Training notebook (Kaggle, two NVIDIA T4 GPUs).

## 0. Configuration

In [ ]:
import os, json, math, time, glob, random, re
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

try:
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)
except Exception as _e:
    print("[WARN] cannot configure the SDP kernels:", _e)

@dataclass
class CFG:
    data_root: str = "/kaggle/input/cfm-tal-shards-1m"
    locked_params_json: str = "/kaggle/input/cfm-tal-shards-1m/Gen1_10Batch_20k/param_names.json"
    out_dir: str = "/kaggle/working"
    shard_glob: str = "**/shard_*.npz"
    resume_input_dir: str = "/kaggle/input/cfm-tal-noisemaker"
    clap_dim: int = 512
    d_token: int = 256
    d_cond:  int = 256
    n_heads: int = 8
    n_layers: int = 8
    n_freq_time: int = 128
    rectified_sigma_min: float = 0.0
    cfg_dropout_rate: float = 0.1
    t_dist: str = "uniform"
    batch_size: int = 128
    lr: float = 1e-4
    weight_decay: float = 0.0
    max_steps: int = 1_000_000
    warmup_steps: int = 10_000
    grad_clip: float = 0.5
    handcrafted_weight: float = 1.0
    hc_test_frac: float = 1.0
    use_ot: bool = True
    ema_decay: float = 0.9999
    val_every: int = 5_000
    val_batch: int = 4096
    sample_val_every: int = 25_000
    sample_val_n: int = 1024
    val_sample_steps: int = 50
    seed: int = 42
    amp: bool = True
    compile: bool = False
    early_stop_enabled: bool = True
    early_stop_warn:     int = 50_000
    early_stop_patience: int = 80_000
    early_stop_min_delta: float = 0.001
    safety_ckpt_every: int = 2_000
    milestone_ckpt_every: int = 50_000
    sample_steps: int = 50
    cfg_strength: float = 2.0

cfg = CFG()
os.makedirs(cfg.out_dir, exist_ok=True)
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed); random.seed(cfg.seed)

if torch.cuda.is_available():
    n_gpu = torch.cuda.device_count()
    print(f"[OK] {n_gpu} GPU: {', '.join(torch.cuda.get_device_name(i) for i in range(n_gpu))}")
    for i in range(n_gpu):
        major, minor = torch.cuda.get_device_capability(i)
        print(f"     GPU{i}: compute capability sm_{major}{minor} "
              f"({'bf16 native' if major>=8 else 'bf16 emulated → fp16 path'})")
else:
    print("[WARN] No CUDA GPU: training on CPU (very slow).")
print(f"[OK] Configuration ready. batch={cfg.batch_size} lr={cfg.lr} "
      f"max_steps={cfg.max_steps} grad_clip={cfg.grad_clip} ema={cfg.ema_decay}")
print(f"     warmup={cfg.warmup_steps} early_stop={'on' if cfg.early_stop_enabled else 'off'} "
      f"(patience={cfg.early_stop_patience})")
print(f"     out_dir={cfg.out_dir!r}")

## 0b. Building the `.npz` shards

In [ ]:
SKIP_BUILD = True

if not SKIP_BUILD:
    raise NotImplementedError(
        "Legacy build disabled: the shards are already built. Set "
        "cfg.data_root to the Kaggle input folder with the shard_*.npz files.")
else:
    import os, json, glob
    cand = [cfg.locked_params_json,
            os.path.join(cfg.data_root, "param_names.json")]
    found = next((p for p in cand if os.path.isfile(p)), None)
    if found:
        with open(found) as fh: param_names = json.load(fh)
        print(f"[OK] param_names from {found}: {len(param_names)} parameters")
    else:
        param_names = None
        print(f"[WARN] param_names.json not found in {cand}.")
        print("       Check that cfg.data_root points to the folder with the shards "
              "and that the generator saved param_names.json next to the .npz files.")

## 1. Parameter encoding in [-1, 1]

In [ ]:
CATEGORICAL = {
    "osc1waveform":      list(np.round(np.linspace(0,1,3),7)),
    "osc2waveform":      list(np.round(np.linspace(0,1,5),7)),
    "filtertype":        list(np.round(np.linspace(0,1,12),7)),
    "lfo1destination":   list(np.round(np.linspace(0,1,8),7)),
    "lfo2destination":   list(np.round(np.linspace(0,1,8),7)),
    "lfo1waveform":      list(np.round(np.linspace(0,1,6),7)),
    "lfo2waveform":      list(np.round(np.linspace(0,1,6),7)),
    "freeaddestination": list(np.round(np.linspace(0,1,6),7)),
}
ORDINAL_49 = ["osc1tune", "osc2tune"]
BINARY = ["lfo1keytrigger", "lfo2keytrigger", "oscsync",
          "chorus1enable", "chorus2enable", "delaysync", "delayx2l", "delayx2r"]
BIPOLAR_NEUTRAL = {
    "lfo1amount", "lfo2amount", "osc1finetune", "osc2finetune",
    "osc1pw", "filtercontour", "freeadamount", "masterdetune",
}

class Param:
    def __init__(self, name, kind, **kw):
        self.name = name; self.kind = kind; self.__dict__.update(kw)
    def __len__(self):
        return len(self.values) if self.kind == "cat" else 1

    def encode(self, v):
        v = float(v)
        if self.kind in ("cont", "bipolar", "ord49"):
            return np.array([2.0 * v - 1.0], np.float32)
        if self.kind == "bin":
            return np.array([1.0 if v >= 0.5 else -1.0], np.float32)
        d = np.abs(np.array(self.values) - v)
        oh = -np.ones(len(self.values), np.float32)
        oh[int(d.argmin())] = 1.0
        return oh

    def decode(self, seg):
        if self.kind in ("cont", "bipolar"):
            return float(np.clip((seg[0] + 1.0) / 2.0, 0, 1))
        if self.kind == "bin":
            return 1.0 if seg[0] >= 0.0 else 0.0
        if self.kind == "ord49":
            v = float(np.clip((seg[0] + 1.0) / 2.0, 0, 1))
            return round(int(round(v * 48)) / 48, 7)
        return self.values[int(np.argmax(seg))]

class ParamSpec:
    def __init__(self, param_names):
        self.params = []
        for n in param_names:
            if n in CATEGORICAL:       self.params.append(Param(n, "cat", values=CATEGORICAL[n]))
            elif n in ORDINAL_49:      self.params.append(Param(n, "ord49"))
            elif n in BINARY:          self.params.append(Param(n, "bin"))
            elif n in BIPOLAR_NEUTRAL: self.params.append(Param(n, "bipolar"))
            else:                      self.params.append(Param(n, "cont"))
        self.layout = []
        i = 0
        for p in self.params:
            k = len(p); self.layout.append((p, slice(i, i + k))); i += k
        self.dim = i

    def encode(self, raw: Dict[str, float]) -> np.ndarray:
        return np.concatenate([p.encode(raw.get(p.name, 0.0)) for p in self.params]).astype(np.float32)

    def encode_matrix(self, P: np.ndarray) -> np.ndarray:
        assert P.shape[1] == len(self.params), f"P ha {P.shape[1]} colonne, spec {len(self.params)}"
        cols = []
        for j, (p, _) in enumerate(self.layout):
            v = P[:, j].astype(np.float32)
            if p.kind in ("cont", "bipolar", "ord49"):
                cols.append((2.0 * v - 1.0)[:, None])
            elif p.kind == "bin":
                cols.append(np.where(v >= 0.5, 1.0, -1.0).astype(np.float32)[:, None])
            else:
                vals = np.asarray(p.values, np.float32)
                idx = np.abs(v[:, None] - vals[None]).argmin(1)
                oh = -np.ones((len(v), len(vals)), np.float32)
                oh[np.arange(len(v)), idx] = 1.0
                cols.append(oh)
        return np.concatenate(cols, 1).astype(np.float32)

    def decode(self, x: np.ndarray) -> Dict[str, float]:
        return {p.name: p.decode(x[sl]) for p, sl in self.layout}

def load_param_names(path):
    try:
        with open(path) as fh:
            raw = json.load(fh)
    except FileNotFoundError:
        raise FileNotFoundError(f"[ERROR] '{path}' not found: run cell 0b.")
    if isinstance(raw, list): return raw
    names = []
    for g in raw.values():
        for p in g:
            if p not in names: names.append(p)
    return names

def roundtrip_test(spec, n=512, seed=0):
    rng = np.random.default_rng(seed); ok = True
    for _ in range(n):
        raw = {}
        for p in spec.params:
            if p.kind == "cat":     raw[p.name] = float(rng.choice(p.values))
            elif p.kind == "bin":   raw[p.name] = float(rng.integers(0, 2))
            elif p.kind == "ord49": raw[p.name] = round(int(rng.integers(0, 49)) / 48, 7)
            else:                   raw[p.name] = float(rng.random())
        dec = spec.decode(spec.encode(raw))
        for k, v in raw.items():
            if abs(dec[k] - v) > 1e-5:
                ok = False; print(f"  [FAIL] {k}: in={v} out={dec[k]}")
    raws = [{p.name: float(rng.random()) for p in spec.params} for _ in range(16)]
    Pm = np.stack([[r[p.name] for p in spec.params] for r in raws]).astype(np.float32)
    a = np.stack([spec.encode(r) for r in raws]); b = spec.encode_matrix(Pm)
    if not np.allclose(a, b, atol=1e-6):
        ok = False; print("  [FAIL] encode and encode_matrix disagree")
    print(f"[{'OK' if ok else 'ERROR'}] ParamSpec roundtrip: {'PASS' if ok else 'FAIL'}")
    assert ok, "ParamSpec roundtrip FAILED: do not start the training."

def report_param_kinds(spec):
    from collections import defaultdict
    by = defaultdict(list)
    for p in spec.params: by[p.kind].append(p.name)
    print("── Parameter types (check suspicious 'cont' entries) ──")
    for k in ("cat","ord49","bin","bipolar","cont"):
        if by[k]:
            print(f"  {k:8s} ({len(by[k])}): {', '.join(by[k])}")
    suspect = [n for n in by["cont"]
               if any(tok in n for tok in ("sync","enable","trigger","keytrig",
                                           "destination","waveform","type","x2"))]
    if suspect:
        print(f"  [⚠️ suspicious: cont, expected bin/cat]: {suspect}")
        print("      → add them to BINARY or CATEGORICAL in cell 6 and run it again.")
    else:
        print("  [OK] no discrete parameter appears misclassified.")

try:
    param_names = load_param_names(cfg.locked_params_json)
    spec = ParamSpec(param_names)
    kinds = {}
    for p in spec.params: kinds[p.kind] = kinds.get(p.kind, 0) + 1
    print(f"[OK] ParamSpec: P_raw={len(param_names)}, dim_CFM={spec.dim} | " +
          "  ".join(f"{k}={v}" for k, v in kinds.items()))
    roundtrip_test(spec)
    report_param_kinds(spec)
except FileNotFoundError as e:
    print(e); spec = None; param_names = []

## 2. Logical constraints (`enforce_audio_logic`)

In [ ]:
_EPS = 0.05

def enforce_audio_logic(params: Dict[str, float]) -> Dict[str, float]:
    def g(key, default=0.0):
        return float(params.get(key, default))

    if g("lfo1destination") == 0.0:
        for p in ["lfo1waveform", "lfo1rate", "lfo1phase", "lfo1keytrigger"]:
            if p in params: params[p] = 0.0
        if "lfo1amount" in params: params["lfo1amount"] = 0.5
    if abs(g("lfo1amount") - 0.5) < _EPS:
        for p in ["lfo1waveform", "lfo1rate", "lfo1phase", "lfo1keytrigger"]:
            if p in params: params[p] = 0.0
        if "lfo1destination" in params: params["lfo1destination"] = 0.0

    if g("lfo2destination") == 0.0:
        for p in ["lfo2waveform", "lfo2rate", "lfo2phase", "lfo2keytrigger"]:
            if p in params: params[p] = 0.0
        if "lfo2amount" in params: params["lfo2amount"] = 0.5
    if abs(g("lfo2amount") - 0.5) < _EPS:
        for p in ["lfo2waveform", "lfo2rate", "lfo2phase", "lfo2keytrigger"]:
            if p in params: params[p] = 0.0
        if "lfo2destination" in params: params["lfo2destination"] = 0.0

    if g("osc1waveform") != 0.5:
        if "osc1pw" in params: params["osc1pw"] = 0.5
    if g("osc2volume") < _EPS and g("osc2fm") < _EPS:
        if "oscsync" in params: params["oscsync"] = 0.0
    if g("osc1volume") < _EPS and g("osc2volume") < _EPS:
        if "ringmodulation" in params: params["ringmodulation"] = 0.0
        if "oscsync" in params:        params["oscsync"]        = 0.0

    if g("filtertype") == 1.0:
        for p in ["filtercutoff", "filterresonance",
                  "filterattack", "filterdecay", "filtersustain", "filterrelease",
                  "filterdrive"]:
            if p in params: params[p] = 0.0
        if "filtercontour" in params: params["filtercontour"] = 0.5
    if abs(g("filtercontour") - 0.5) < _EPS:
        for p in ["filterattack", "filterdecay", "filtersustain", "filterrelease"]:
            if p in params: params[p] = 0.0

    if g("freeaddestination") == 0.0:
        for p in ["freeadattack", "freeaddecay"]:
            if p in params: params[p] = 0.0
        if "freeadamount" in params: params["freeadamount"] = 0.5
    if abs(g("freeadamount") - 0.5) < _EPS:
        for p in ["freeadattack", "freeaddecay"]:
            if p in params: params[p] = 0.0
        if "freeaddestination" in params: params["freeaddestination"] = 0.0

    if g("reverbwet") < _EPS:
        for p in ["reverbdecay", "reverbpredelay", "reverbhighcut", "reverblowcut"]:
            if p in params: params[p] = 0.0
    if g("delaywet") < _EPS:
        for p in ["delaytime", "delaysync", "delayhighshelf", "delaylowshelf",
                  "delayfeedback", "delayx2l", "delayx2r"]:
            if p in params: params[p] = 0.0

    return params

## 3. Dataset and minibatch OT coupling

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from scipy.optimize import linear_sum_assignment
try:
    from threadpoolctl import threadpool_limits
except ImportError:
    from contextlib import contextmanager
    @contextmanager
    def threadpool_limits(limits=None): yield

class PresetDataset(Dataset):
    _shared = None

    def __init__(self, cfg, spec, split="train", share=True):
        assert split in ("train", "val", "test_hc")
        if share and PresetDataset._shared is not None:
            sh = PresetDataset._shared
            self.clap, self.x, self.source = sh["clap"], sh["x"], sh["source"]
            self._build_split(cfg, split)
            print(f"[OK] Split '{split}' (arrays condivisi, no re-load): "
                  f"{len(self.ids):,} samples")
            return
        shards = sorted(glob.glob(os.path.join(cfg.data_root, cfg.shard_glob),
                                   recursive=True))
        if not shards:
            raise FileNotFoundError(
                f"[ERROR] No shard found with '{cfg.shard_glob}' in "
                f"'{cfg.data_root}'. Check that the zip file on Kaggle was "
                f"uploaded as a dataset and attached as input.")
        print(f"[OK] Found {len(shards)} shards in {len(set(os.path.dirname(s) for s in shards))} subfolders")

        param_files = sorted(set(os.path.join(os.path.dirname(s), "param_names.json")
                                  for s in shards
                                  if os.path.exists(os.path.join(os.path.dirname(s), "param_names.json"))))
        if len(param_files) > 1:
            ref = json.load(open(param_files[0]))
            for pf in param_files[1:]:
                other = json.load(open(pf))
                if other != ref:
                    raise RuntimeError(f"[ERROR] param_names mismatch:\n"
                                       f"  {param_files[0]}\n  ≠ {pf}\n"
                                       f"All shards must use the same parameter order.")
            print(f"[OK] param_names.json identical in {len(param_files)} subfolders")

        C, X, S = [], [], []
        total_bytes = 0
        for path in shards:
            try:
                d = np.load(path)
            except Exception as e:
                print(f"  [WARN] '{path}': {e}: skipped."); continue
            clap_arr = d["clap"].astype(np.float16)
            P        = d["params"].astype(np.float32)
            if len(clap_arr) != len(P):
                print(f"  [ERROR] '{os.path.basename(path)}': clap {len(clap_arr)} "
                      f"≠ params {len(P)}: skipped."); continue
            if P.shape[1] != len(param_names):
                print(f"  [WARN] '{path}': P.shape[1]={P.shape[1]} ≠ "
                      f"{len(param_names)}: skipped."); continue
            x_enc = spec.encode_matrix(P).astype(np.float16)
            X.append(x_enc); C.append(clap_arr)
            S.append(d["source"].astype(np.int64) if "source" in d else
                     np.zeros(len(P), np.int64))
            total_bytes += clap_arr.nbytes + x_enc.nbytes
            print(f"  [OK]  {os.path.relpath(path, cfg.data_root):50s}  N={len(P):>6d}")
        if not C:
            raise RuntimeError("[ERROR] No valid shard.")
        self.clap   = np.concatenate(C)
        self.x      = np.concatenate(X)
        self.source = np.concatenate(S)
        del C, X, S

        print(f"[OK] Dataset totale: N={len(self.x):,}  RAM={total_bytes/1e9:.2f} GB "
              f"(clap fp16 + encoded fp16)")
        m, s = self.x.astype(np.float32).mean(), self.x.astype(np.float32).std()
        print(f"[OK] Encoded x: mean={m:+.3f}  std={s:.3f}  "
              f"min={self.x.min():.2f}  max={self.x.max():.2f}")

        if share:
            PresetDataset._shared = {"clap": self.clap, "x": self.x,
                                      "source": self.source}

        self._build_split(cfg, split, verbose=True)

    def _build_split(self, cfg, split, verbose=False):
        rng = np.random.default_rng(cfg.seed)
        syn = np.where(self.source == 0)[0]
        hc  = np.where(self.source == 1)[0]
        syn_perm = rng.permutation(syn)
        val_ids     = syn_perm[: int(0.05 * len(syn))]
        test_hc_ids = rng.permutation(hc)[: int(cfg.hc_test_frac * len(hc))]
        held = np.zeros(len(self.x), bool)
        held[val_ids] = True; held[test_hc_ids] = True
        train_ids = np.where(~held)[0]
        train_ids = train_ids[self.source[train_ids] == 0]

        self.train_ids, self.val_ids, self.test_hc_ids = train_ids, val_ids, test_hc_ids
        self.ids = {"train": train_ids, "val": val_ids, "test_hc": test_hc_ids}[split]
        if verbose:
            n_hc_train = int(self.source[train_ids].sum())
            print(f"[OK] Split '{split}': {len(self.ids):,} | train={len(train_ids):,} "
                  f"(hc={n_hc_train}) | val={len(val_ids):,} | "
                  f"test_hc={len(test_hc_ids):,} (OOD)")
            if n_hc_train != 0:
                print(f"  [WARN] hc in training: {n_hc_train}. Expected 0.")

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        j = self.ids[i]
        return (torch.from_numpy(self.clap[j].astype(np.float32)),
                torch.from_numpy(self.x[j].astype(np.float32)))

    def sampler_weights(self, hc):
        return np.ones(len(self.train_ids), np.float32)

def ot_collate(batch):
    clap = torch.stack([b[0] for b in batch])
    x1   = torch.stack([b[1] for b in batch])
    x0   = torch.randn_like(x1)
    cost = torch.cdist(x0, x1).numpy()
    with threadpool_limits(limits=1):
        r, c = linear_sum_assignment(cost)
    return clap[c], x1[c], x0[r]

def plain_collate(batch):
    clap = torch.stack([b[0] for b in batch])
    x1   = torch.stack([b[1] for b in batch])
    return clap, x1, torch.randn_like(x1)

## 4. Vector field: DiT-style Transformer, one token per parameter

In [ ]:
class TimeEmbed(nn.Module):
    def __init__(self, d_cond, n_freq=128, max_freq=1000.0):
        super().__init__()
        freqs = torch.exp(torch.linspace(math.log(1.0), math.log(max_freq), n_freq))
        self.register_buffer("freqs", freqs, persistent=False)
        self.mlp = nn.Sequential(nn.Linear(2 * n_freq, d_cond), nn.SiLU(), nn.Linear(d_cond, d_cond))
    def forward(self, t):
        ang = 2 * math.pi * t * self.freqs[None]
        return self.mlp(torch.cat([ang.sin(), ang.cos()], -1))

class DiTBlock(nn.Module):
    def __init__(self, d, n_heads, d_cond):
        super().__init__()
        self.n1 = nn.LayerNorm(d, elementwise_affine=False)
        self.n2 = nn.LayerNorm(d, elementwise_affine=False)
        self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, d * 4), nn.GELU(), nn.Linear(d * 4, d))
        self.ada = nn.Sequential(nn.SiLU(), nn.Linear(d_cond, 6 * d))
        nn.init.zeros_(self.ada[-1].weight); nn.init.zeros_(self.ada[-1].bias)
    def forward(self, x, z):
        sa, ga, ha, sf, gf, hf = self.ada(z).unsqueeze(1).chunk(6, -1)
        h = self.n1(x) * (1 + ga) + sa
        a, _ = self.attn(h, h, h)
        x = x + ha * a
        h = self.n2(x) * (1 + gf) + sf
        x = x + hf * self.ff(h)
        return x

class FinalLayer(nn.Module):
    def __init__(self, d, d_cond):
        super().__init__()
        self.norm = nn.LayerNorm(d, elementwise_affine=False)
        self.ada = nn.Sequential(nn.SiLU(), nn.Linear(d_cond, 2 * d))
        nn.init.zeros_(self.ada[-1].weight); nn.init.zeros_(self.ada[-1].bias)
    def forward(self, x, z):
        s, g = self.ada(z).unsqueeze(1).chunk(2, -1)
        return self.norm(x) * (1 + g) + s

class VectorField(nn.Module):
    def __init__(self, spec, d_token, n_heads, n_layers, clap_dim, d_cond, n_freq_time=128):
        super().__init__()
        self.spec, self.d_token = spec, d_token
        n_params = len(spec.params)
        self.in_p  = nn.ModuleList([nn.Linear(len(p), d_token) for p, _ in spec.layout])
        self.out_p = nn.ModuleList([nn.Linear(d_token, len(p)) for p, _ in spec.layout])
        for lin in self.out_p:
            nn.init.zeros_(lin.weight); nn.init.zeros_(lin.bias)
        self.p_emb = nn.Embedding(n_params, d_token)

        self.t_emb    = TimeEmbed(d_cond, n_freq_time)
        self.clap_mlp = nn.Sequential(nn.Linear(clap_dim, d_cond), nn.SiLU(), nn.Linear(d_cond, d_cond))
        self.cfg_token = nn.Parameter(torch.randn(1, clap_dim) / math.sqrt(clap_dim))

        self.blocks = nn.ModuleList([DiTBlock(d_token, n_heads, d_cond) for _ in range(n_layers)])
        self.final  = FinalLayer(d_token, d_cond)

    def apply_dropout(self, c, rate):
        if rate == 0: return c
        keep = (torch.rand(c.shape[0], 1, device=c.device) > rate)
        return torch.where(keep, c, self.cfg_token.expand(c.shape[0], -1))

    def _tokenize(self, x):
        toks = torch.stack([self.in_p[i](x[:, sl]) for i, (_, sl) in enumerate(self.spec.layout)], dim=1)
        return toks + self.p_emb(torch.arange(len(self.spec.params), device=x.device))

    def forward(self, x, t, cond):
        B = x.shape[0]
        if cond is None:
            cond = self.cfg_token.expand(B, -1)
        z = self.t_emb(t) + self.clap_mlp(cond)
        toks = self._tokenize(x)
        for blk in self.blocks:
            toks = blk(toks, z)
        toks = self.final(toks, z)
        return torch.cat([self.out_p[i](toks[:, i]) for i in range(len(self.spec.layout))], dim=1)

## 5. Rectified path, RK4 sampling with CFG, restart from t*

In [ ]:
def rectified_path(x0, x1, t, sigma_min):
    return x0 * (1 - t) * (1 - sigma_min) + x1 * t

def sample_t(n, device, dist="uniform"):
    if dist == "logit_normal":
        return torch.sigmoid(torch.randn(n, 1, device=device))
    return torch.rand(n, 1, device=device)

def call_cfg(vf, x, t, cond, s):
    if s == 1.0 or cond is None:
        return vf(x, t, cond)
    yc = vf(x, t, cond); yu = vf(x, t, None)
    return (1 - s) * yu + s * yc

def rk4_step(vf, x, t, dt, cond, s):
    f = lambda xx, tt: call_cfg(vf, xx, tt, cond, s)
    k1 = f(x, t); k2 = f(x + dt * k1 / 2, t + dt / 2)
    k3 = f(x + dt * k2 / 2, t + dt / 2); k4 = f(x + dt * k3, t + dt)
    return x + (dt / 6) * (k1 + 2 * k2 + 2 * k3 + k4)

@torch.no_grad()
def sample(vf, cond, n_dim, steps, s, device, n=None, x0=None):
    n = cond.shape[0] if cond is not None else n
    x = torch.randn(n, n_dim, device=device) if x0 is None else x0.clone()
    t = torch.zeros(n, 1, device=device); dt = 1.0 / steps
    for _ in range(steps):
        x = rk4_step(vf, x, t, dt, cond, s); t = t + dt
    return x

@torch.no_grad()
def sample_from_init(vf, x_init, cond, t_star, steps, s, device):
    assert 0.0 < t_star < 1.0
    x0 = torch.randn_like(x_init)
    x  = (1 - t_star) * x0 + t_star * x_init
    n  = x.shape[0]
    t  = torch.full((n, 1), t_star, device=device)
    dt = (1.0 - t_star) / steps
    for _ in range(steps):
        x = rk4_step(vf, x, t, dt, cond, s); t = t + dt
    return x

@torch.no_grad()
def sample_best_of_n(vf, cond, n_dim, N, steps, s, device):
    B = cond.shape[0]
    cond_rep = cond.repeat_interleave(N, dim=0)
    x = sample(vf, cond_rep, n_dim, steps, s, device)
    return x.view(B, N, n_dim).transpose(0, 1).contiguous()

## 6. Training: EMA, warmup and cosine schedule, validation loss

In [ ]:
import threading, shutil, subprocess

_KAGGLE_USER = os.environ.get("KAGGLE_USERNAME", "")
_KAGGLE_SLUG = "cfm-tal-noisemaker"
_bg_upload   = None

def _test_kaggle_api():
    try:
        r = subprocess.run(["kaggle", "datasets", "list", "--mine", "--max-size", "1"],
                           capture_output=True, text=True, timeout=30)
    except Exception as e:
        print(f"[WARN] Kaggle API not available: {e}"); return False
    if r.returncode == 0:
        print("[OK] Kaggle API disponibile."); return True
    print(f"[WARN] Kaggle API: {r.stderr.strip()[:150]} → checkpoint solo locali.")
    return False

def _kaggle_push(out_dir, label, files_to_push=None):
    upload_dir = os.path.join(out_dir, "_upload_tmp")
    os.makedirs(upload_dir, exist_ok=True)
    for old in glob.glob(os.path.join(upload_dir, "*.pt")): os.remove(old)
    if files_to_push is None:
        pts = sorted(glob.glob(os.path.join(out_dir, "cfm_*.pt")))
    else:
        pts = [os.path.join(out_dir, f) for f in files_to_push
               if os.path.exists(os.path.join(out_dir, f))]
    for f in pts: shutil.copy2(f, upload_dir)
    with open(os.path.join(upload_dir, "dataset-metadata.json"), "w") as fh:
        json.dump({"title": "CFM TAL-NoiseMaker",
                   "id": f"{_KAGGLE_USER}/{_KAGGLE_SLUG}",
                   "licenses": [{"name": "CC0-1.0"}]}, fh, indent=2)
    r = subprocess.run(["kaggle", "datasets", "version", "-p", upload_dir, "-m", label,
                        "--dir-mode", "zip"], capture_output=True, text=True, timeout=900)
    if r.returncode != 0:
        r = subprocess.run(["kaggle", "datasets", "create", "-p", upload_dir,
                            "--dir-mode", "zip"], capture_output=True, text=True, timeout=900)
    print(f"  [KAGGLE {'✓' if r.returncode == 0 else '✗'}] '{label}' ({len(pts)} files)"
          + ("" if r.returncode == 0 else f": {r.stderr.strip()[:200]}"))

def _async_push(out_dir, label, files_to_push=None):
    global _bg_upload
    if _bg_upload and _bg_upload.is_alive():
        print(f"  [INFO] Previous upload still running: '{label}' included in the next one."); return
    _bg_upload = threading.Thread(target=_kaggle_push,
                                  args=(out_dir, label, files_to_push), daemon=False)
    _bg_upload.start()

def _wait_upload():
    if _bg_upload and _bg_upload.is_alive():
        print("[OK] Waiting for the last Kaggle upload..."); _bg_upload.join(timeout=900)

class EMA:
    def __init__(self, module, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float()
                       for k, v in module.state_dict().items()}
    @torch.no_grad()
    def update(self, module):
        for k, v in module.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()
    def cpu_state(self):
        return {k: v.cpu() for k, v in self.shadow.items()}
    def load_state(self, state):
        self.shadow = {k: v.to(next(iter(self.shadow.values())).device)
                       for k, v in state.items()}

def _amp_tools(device_type="cuda"):
    try:
        scaler = torch.amp.GradScaler(device_type, enabled=True)
        ac = lambda dtype, enabled: torch.amp.autocast(device_type, dtype=dtype, enabled=enabled)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=True)
        ac = lambda dtype, enabled: torch.cuda.amp.autocast(dtype=dtype, enabled=enabled)
    return scaler, ac

def find_resume_checkpoint(cfg):
    candidates_local = glob.glob(os.path.join(cfg.out_dir, "cfm_*.pt"))
    candidates_input = (glob.glob(os.path.join(cfg.resume_input_dir, "cfm_*.pt"))
                        if os.path.isdir(cfg.resume_input_dir) else [])
    local_latest = os.path.join(cfg.out_dir, "cfm_latest.pt")
    if os.path.exists(local_latest):
        return local_latest
    input_latest = os.path.join(cfg.resume_input_dir, "cfm_latest.pt")
    if os.path.exists(input_latest):
        return input_latest
    numbered = []
    for c in candidates_local + candidates_input:
        m = re.match(r"cfm_step(\d+)\.pt", os.path.basename(c))
        if m: numbered.append((int(m.group(1)), c))
    if numbered:
        return max(numbered)[1]
    return None

def train(cfg):
    kaggle_ok = _test_kaggle_api()
    _param_names = load_param_names(cfg.locked_params_json)
    _spec = ParamSpec(_param_names)
    roundtrip_test(_spec, n=64)

    ds = PresetDataset(cfg, _spec, "train")
    w = ds.sampler_weights(cfg.handcrafted_weight)
    samp = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double),
                                 num_samples=len(ds), replacement=True)
    dl = DataLoader(ds, batch_size=cfg.batch_size, sampler=samp,
                    num_workers=4,
                    persistent_workers=True,
                    prefetch_factor=2,
                    pin_memory=True, drop_last=True,
                    collate_fn=ot_collate if cfg.use_ot else plain_collate)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    vf = VectorField(_spec, cfg.d_token, cfg.n_heads, cfg.n_layers,
                     cfg.clap_dim, cfg.d_cond, cfg.n_freq_time).to(device)
    n_gpu = torch.cuda.device_count()
    raw_vf = vf
    if n_gpu > 1:
        print(f"[OK] DataParallel su {n_gpu} GPU"); vf = nn.DataParallel(vf)
    elif cfg.compile:
        vf = torch.compile(vf); print("[OK] torch.compile enabled")
    module = raw_vf
    print(f"[OK] VectorField DiT: {sum(p.numel() for p in module.parameters())/1e6:.2f}M "
          f"params (d_token={cfg.d_token}, layers={cfg.n_layers}, d_cond={cfg.d_cond})")

    ema = EMA(module, cfg.ema_decay)

    opt = torch.optim.AdamW(vf.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    def lr_lambda(s):
        if s < cfg.warmup_steps: return (s + 1) / max(1, cfg.warmup_steps)
        p = (s - cfg.warmup_steps) / max(1, cfg.max_steps - cfg.warmup_steps)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    if torch.cuda.is_available():
        major, _ = torch.cuda.get_device_capability(0)
        bf16_native = (major >= 8)
    else:
        bf16_native = False
    use_fp16 = cfg.amp and not bf16_native
    dtype = torch.float16 if use_fp16 else torch.bfloat16
    print(f"[OK] AMP: {'fp16+GradScaler' if use_fp16 else ('bf16' if cfg.amp else 'fp32')} "
          f"(bf16_native={bf16_native})")
    scaler, autocast = _amp_tools()

    val_ds = PresetDataset(cfg, _spec, "val")
    g = torch.Generator().manual_seed(cfg.seed)
    nv = min(cfg.val_batch, len(val_ds))
    vi = torch.randperm(len(val_ds), generator=g)[:nv]
    v_clap = torch.stack([val_ds[i][0] for i in vi]).to(device)
    v_x1   = torch.stack([val_ds[i][1] for i in vi]).to(device)
    v_x0   = torch.randn(v_x1.shape, generator=g).to(device)
    v_t    = torch.rand(nv, 1, generator=g).to(device)

    @torch.no_grad()
    def val_loss():
        vf.eval()
        x_t = rectified_path(v_x0, v_x1, v_t, cfg.rectified_sigma_min)
        with autocast(dtype, cfg.amp):
            l = (vf(x_t, v_t, v_clap) - (v_x1 - v_x0)).square().mean()
        vf.train()
        return l.item()

    step = 0
    best_val = float("inf")
    best_val_step = 0
    resume_path = find_resume_checkpoint(cfg)
    if resume_path:
        print(f"[RESUME] Checkpoint found: {resume_path}")
        ck = torch.load(resume_path, map_location=device)
        module.load_state_dict(ck["vf"])
        ema.shadow = {k: v.to(device) if v.dtype.is_floating_point else v
                       for k, v in ck["ema"].items()}
        if "opt" in ck:    opt.load_state_dict(ck["opt"])
        if "sched" in ck:  sched.load_state_dict(ck["sched"])
        if "scaler" in ck and ck["scaler"] is not None and use_fp16:
            scaler.load_state_dict(ck["scaler"])
        step = ck.get("step", 0)
        best_val = ck.get("best_val", float("inf"))
        best_val_step = ck.get("best_val_step", step)
        if "rng_torch" in ck:
            try:
                torch.set_rng_state(ck["rng_torch"].cpu() if hasattr(ck["rng_torch"], "cpu")
                                    else ck["rng_torch"])
            except Exception as e: print(f"  [WARN] RNG torch: {e}")
        if "rng_torch_cuda" in ck and torch.cuda.is_available():
            try: torch.cuda.set_rng_state_all(ck["rng_torch_cuda"])
            except Exception as e: print(f"  [WARN] RNG cuda: {e}")
        if "rng_numpy" in ck:
            try: np.random.set_state(ck["rng_numpy"])
            except Exception as e: print(f"  [WARN] RNG numpy: {e}")
        if "rng_python" in ck:
            try: random.setstate(ck["rng_python"])
            except Exception as e: print(f"  [WARN] RNG python: {e}")
        print(f"[RESUME] step={step}  best_val={best_val:.4f}  best@{best_val_step}")
    else:
        print(f"[OK] No checkpoint found: training from scratch")

    print(f"[OK] Training: batch={cfg.batch_size} steps={cfg.max_steps} "
          f"(from step {step}) AMP {dtype} OT={'on' if cfg.use_ot else 'off'} "
          f"t~{cfg.t_dist} val n={nv} val-S every {cfg.sample_val_every}")
    print(f"     early-stop: {'on' if cfg.early_stop_enabled else 'off'} "
          f"(patience={cfg.early_stop_patience}, warn={cfg.early_stop_warn}, "
          f"min_delta={cfg.early_stop_min_delta})")

    def save_checkpoint(path):
        torch.save({
            "vf": module.state_dict(),
            "ema": ema.cpu_state(),
            "opt": opt.state_dict(),
            "sched": sched.state_dict(),
            "scaler": scaler.state_dict() if use_fp16 else None,
            "step": step,
            "best_val": best_val,
            "best_val_step": best_val_step,
            "rng_torch": torch.get_rng_state(),
            "rng_torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            "rng_numpy": np.random.get_state(),
            "rng_python": random.getstate(),
            "cfg": cfg.__dict__,
            "param_names": _param_names,
        }, path)

    t0 = time.time(); n_bad = 0; warned_plateau = False
    while step < cfg.max_steps:
        for clap, x1, x0 in dl:
            clap, x1, x0 = clap.to(device), x1.to(device), x0.to(device)
            t = sample_t(x1.shape[0], device, cfg.t_dist)
            x_t = rectified_path(x0, x1, t, cfg.rectified_sigma_min)
            target = x1 - x0
            z = module.apply_dropout(clap, cfg.cfg_dropout_rate)

            opt.zero_grad(set_to_none=True)
            with autocast(dtype, cfg.amp):
                loss = (vf(x_t, t, z) - target).square().mean()
            if not torch.isfinite(loss):
                n_bad += 1
                if n_bad <= 5 or n_bad % 100 == 0:
                    print(f"  [WARN] non-finite loss @step {step}: batch skipped (#{n_bad})")
                opt.zero_grad(set_to_none=True)
                sched.step(); step += 1
                continue
            if use_fp16:
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                gnorm = nn.utils.clip_grad_norm_(vf.parameters(), cfg.grad_clip)
                scaler.step(opt); scaler.update()
            else:
                loss.backward()
                gnorm = nn.utils.clip_grad_norm_(vf.parameters(), cfg.grad_clip)
                opt.step()
            sched.step(); ema.update(module); step += 1

            if step % 500 == 0:
                print(f"  step {step:>7d}/{cfg.max_steps}  loss={loss.item():.4f}  "
                      f"gn={float(gnorm):.2f}  lr={sched.get_last_lr()[0]:.2e}  "
                      f"{int(time.time()-t0)}s")

            if step % cfg.val_every == 0:
                vl = val_loss(); flag = ""
                if vl < best_val * (1 - cfg.early_stop_min_delta):
                    best_val = vl; best_val_step = step; flag = "  ← best"
                    warned_plateau = False
                steps_since_best = step - best_val_step
                if cfg.early_stop_enabled and steps_since_best >= cfg.early_stop_warn and not warned_plateau:
                    print(f"  [WARN-STOP] val_fm plateau for {steps_since_best} steps "
                          f"(soft warn; hard stop a {cfg.early_stop_patience})")
                    warned_plateau = True
                print(f"  [VAL] step {step:>7d}  val_fm={vl:.4f}  "
                      f"(best={best_val:.4f} @{best_val_step}){flag}")
                if cfg.early_stop_enabled and steps_since_best >= cfg.early_stop_patience:
                    print(f"  [STOP] Early stop at step {step}: no improvement "
                          f"for {steps_since_best} steps. Best val_fm={best_val:.4f} @{best_val_step}")
                    break

            if cfg.sample_val_every and step % cfg.sample_val_every == 0:
                vf.eval()
                try:
                    with torch.no_grad():
                        ns = min(cfg.sample_val_n, nv)
                        xs = sample(module, v_clap[:ns], v_x1.shape[1], cfg.val_sample_steps,
                                    cfg.cfg_strength, device, x0=v_x0[:ns])
                        smse = (xs - v_x1[:ns]).square().mean().item()
                    print(f"  [VAL-S] step {step:>7d}  sampled_param_mse={smse:.4f}  "
                          f"(n={ns}, {cfg.val_sample_steps} step, s={cfg.cfg_strength})")
                except Exception as _e:
                    print(f"  [WARN] sampled validation skipped @step {step}: "
                          f"{type(_e).__name__}: {_e}")
                vf.train()

            if step % cfg.safety_ckpt_every == 0:
                save_checkpoint(os.path.join(cfg.out_dir, "cfm_latest.pt"))

            if step % cfg.milestone_ckpt_every == 0:
                mpath = os.path.join(cfg.out_dir, f"cfm_step{step}.pt")
                save_checkpoint(mpath)
                print(f"  [OK] Milestone: {mpath}")
                if kaggle_ok:
                    _async_push(cfg.out_dir, f"step-{step}",
                                files_to_push=[f"cfm_step{step}.pt", "cfm_latest.pt"])

            if step >= cfg.max_steps: break
        else:
            continue
        break

    final = os.path.join(cfg.out_dir, "cfm_final.pt")
    save_checkpoint(final)
    print(f"\n[OK] Training completed in {(time.time()-t0)/3600:.2f}h "
          f"({step} step totali) → {final}")
    print(f"[OK] Best val_fm: {best_val:.4f} @ step {best_val_step}")
    if kaggle_ok:
        _wait_upload()
        _kaggle_push(cfg.out_dir, "final",
                     files_to_push=["cfm_final.pt", "cfm_latest.pt"])

def launch():
    if not torch.cuda.is_available():
        raise RuntimeError("[ERROR] No CUDA GPU available.")
    print(f"[OK] Starting training on {torch.cuda.device_count()} GPUs.")
    train(cfg)

## 6a. Pre-flight check

In [ ]:
import glob as _glob
_shards = _glob.glob(os.path.join(cfg.data_root, "**/shard_*.npz"), recursive=True)
_pnames = _glob.glob(os.path.join(cfg.data_root, "**/param_names.json"), recursive=True)
print(f"[CHECK] data_root = {cfg.data_root}")
print(f"[CHECK] shards found: {len(_shards)}")
for s in _shards[:3]: print("   ", s)
if len(_shards) > 3: print(f"    ... and {len(_shards)-3}")
print(f"[CHECK] param_names.json files found: {len(_pnames)}")
for p in _pnames[:3]: print("   ", p)

assert len(_shards) > 0, "❌ NO SHARDS: fix cfg.data_root"
assert len(_pnames) > 0, "❌ NO param_names.json: fix the paths"

if not os.path.exists(cfg.locked_params_json):
    print(f"[FIX] cfg.locked_params_json does not exist, using: {_pnames[0]}")
    cfg.locked_params_json = _pnames[0]
print(f"[OK] locked_params_json = {cfg.locked_params_json}")

_n_total = 0
for s in _shards:
    with np.load(s) as d:
        _n_total += len(d["params"])
print(f"[OK] Estimated total samples: {_n_total:,}")

In [ ]:
DRY_RUN = False

if not DRY_RUN:
    print("[SKIP] DRY_RUN=False: dry run skipped. The actual training starts in cell 6b.")
else:
    _REAL = dict(max_steps=cfg.max_steps,
                 milestone_ckpt_every=cfg.milestone_ckpt_every,
                 safety_ckpt_every=cfg.safety_ckpt_every,
                 val_every=cfg.val_every,
                 sample_val_every=cfg.sample_val_every,
                 early_stop_enabled=cfg.early_stop_enabled)
    cfg.max_steps            = 2000
    cfg.milestone_ckpt_every = 1000
    cfg.safety_ckpt_every    = 500
    cfg.val_every            = 500
    cfg.sample_val_every     = 1000
    cfg.early_stop_enabled   = False
    try: PresetDataset._shared = None
    except Exception: pass

    print("="*70)
    print("DRY RUN: 2000 steps. Checks: loading, no NaN, val_fm~0.4, checkpoint, upload.")
    print("="*70)
    train(cfg)

    for k, v in _REAL.items(): setattr(cfg, k, v)
    for _f in glob.glob(os.path.join(cfg.out_dir, "cfm_*.pt")):
        os.remove(_f); print(f"  [CLEANUP] removed {os.path.basename(_f)}")
    try: PresetDataset._shared = None
    except Exception: pass
    print("\n[OK] Dry run finished, settings restored, working directory cleaned.")
    print(f"     max_steps={cfg.max_steps}  milestone_every={cfg.milestone_ckpt_every}")
    print("     → now set DRY_RUN=False and use Save Version for the actual training.")

## 6b. Training

In [ ]:
launch()

## 7. Inference

In [ ]:
def _load_vf(ckpt_path, use_ema=True, device="cuda"):
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    c = CFG(**{k: v for k, v in ck["cfg"].items() if k in CFG.__dataclass_fields__})
    _spec = ParamSpec(ck["param_names"])
    vf = VectorField(_spec, c.d_token, c.n_heads, c.n_layers,
                     c.clap_dim, c.d_cond, getattr(c, "n_freq_time", 128)).to(device)
    state = ck["ema"] if (use_ema and "ema" in ck) else ck["vf"]
    vf.load_state_dict(state); vf.eval()
    print(f"[OK] Checkpoint: {ckpt_path} | pesi={'EMA' if use_ema and 'ema' in ck else 'raw'}")
    return vf, c, _spec

def predict_from_audio(ckpt, clap_audio, steps=None, s=None, use_ema=True):
    vf, c, _spec = _load_vf(ckpt, use_ema)
    cond = torch.from_numpy(np.asarray(clap_audio, np.float32)).cuda()
    x = sample(vf, cond, _spec.dim, steps or c.sample_steps, s or c.cfg_strength, "cuda").cpu().numpy()
    return [enforce_audio_logic(_spec.decode(x[i])) for i in range(len(x))]

def predict_edit(ckpt, clap_target, init_params: Dict[str, float],
                 t_star=0.6, steps=None, s=None, use_ema=True):
    vf, c, _spec = _load_vf(ckpt, use_ema)
    cond = torch.from_numpy(np.asarray(clap_target, np.float32)).cuda()
    x_init = torch.from_numpy(_spec.encode(init_params)).cuda().unsqueeze(0).expand(cond.shape[0], -1)
    x = sample_from_init(vf, x_init, cond, t_star,
                         steps or c.sample_steps, s or c.cfg_strength, "cuda").cpu().numpy()
    out = [enforce_audio_logic(_spec.decode(x[i])) for i in range(len(x))]
    moved = [np.mean([abs(o[k] - float(init_params.get(k, 0.0))) for k in o]) for o in out]
    print(f"[OK] t*={t_star} | mean L1 shift from the initial preset: {np.mean(moved):.4f}")
    return out

## 8. Diagnostics in parameter space

In [ ]:
def eval_params(ckpt, split="val", n=2000, steps=None, s=None, use_ema=True):
    ds = PresetDataset(cfg, spec, split)
    ids = ds.ids[:n]
    if len(ids) == 0:
        print(f"[WARN] split '{split}' is empty."); return
    print(f"[OK] Parameter diagnostics on {len(ids)} samples ({split})...")
    preds = predict_from_audio(ckpt, ds.clap[ids], steps=steps, s=s, use_ema=use_ema)

    cont, bip_mae, bip_sign, acc, tune = [], [], [], [], []
    per_param = {p.name: [] for p, _ in spec.layout}
    for k, i in enumerate(ids):
        tgt, pr = spec.decode(ds.x[i]), preds[k]
        for p, _ in spec.layout:
            err = abs(pr[p.name] - tgt[p.name]); per_param[p.name].append(err)
            if p.kind == "cont": cont.append(err)
            elif p.kind == "bipolar":
                bip_mae.append(err)
                ts, ps = tgt[p.name] - 0.5, pr[p.name] - 0.5
                if abs(ts) > 0.05: bip_sign.append(1.0 if ts * ps >= 0 else 0.0)
            elif p.kind == "cat": acc.append(1.0 if pr[p.name] == tgt[p.name] else 0.0)
            elif p.kind == "ord49": tune.append(err)

    print(f"\n── Diagnostics '{split}' " + "─" * 38)
    print(f"  Continuous MAE   : {np.mean(cont):.4f}  (n={len(cont)})")
    if bip_mae:  print(f"  Bipolar MAE      : {np.mean(bip_mae):.4f}")
    if bip_sign: print(f"  Bipolar sign acc : {np.mean(bip_sign):.3f}  (neutral values excluded)")
    if acc:      print(f"  Categorical acc  : {np.mean(acc):.3f}")
    if tune:     print(f"  Tune(49) MAE     : {np.mean(tune):.4f}")
    worst = sorted(per_param.items(), key=lambda kv: -np.mean(kv[1]))[:10]
    print("  10 worst parameters (MAE):")
    for nme, errs in worst: print(f"    {nme:22s} {np.mean(errs):.4f}")
    print("─" * 62)

## 8c. Audio-space evaluation (multi-scale spectral distance)

In [ ]:
VST_PATH    = "/Library/Audio/Plug-Ins/VST3/TAL-NoiseMaker.vst3"
RENDER_SR   = 48000
RENDER_DUR  = 4.0
NOTE        = (60, 100, 0.0, 2.0)

def normalize_name(text):
    return text.lower().replace("_", "").replace(" ", "")

def make_engine(vst_path=VST_PATH, sr=RENDER_SR):
    import dawdreamer as daw
    eng = daw.RenderEngine(sr, 512)
    synth = eng.make_plugin_processor("synth", vst_path)
    eng.load_graph([(synth, [])])
    idx_map = {normalize_name(p["name"]): p["index"]
               for p in synth.get_parameters_description()}
    return eng, synth, idx_map

def render_preset(eng, synth, idx_map, params: Dict[str, float],
                  sr=RENDER_SR, dur=RENDER_DUR, note=NOTE, master=0.8):
    for n, v in params.items():
        key = normalize_name(n)
        if key in idx_map: synth.set_parameter(idx_map[key], float(v))
    if "mastervolume" in idx_map: synth.set_parameter(idx_map["mastervolume"], master)
    synth.clear_midi()
    synth.add_midi_note(*[int(note[0]), int(note[1]), float(note[2]), float(note[3])])
    eng.render(dur)
    return eng.get_audio().mean(axis=0).astype(np.float32)

def mss_distance(a: np.ndarray, b: np.ndarray, ffts=(2048,1024,512,256,128,64)) -> float:
    ta, tb = torch.from_numpy(a).float(), torch.from_numpy(b).float()
    L = min(len(ta), len(tb)); ta, tb = ta[:L], tb[:L]
    tot = 0.0
    for n in ffts:
        win = torch.hann_window(n)
        Sa = torch.stft(ta, n, n // 4, window=win, return_complex=True).abs()
        Sb = torch.stft(tb, n, n // 4, window=win, return_complex=True).abs()
        tot += (Sa - Sb).abs().mean() + (torch.log(Sa + 1e-5) - torch.log(Sb + 1e-5)).abs().mean()
    return float(tot / len(ffts))

def eval_audio_space(ckpt, clap_emb: np.ndarray, target_wavs: List[str],
                     N=4, steps=None, s=None, use_ema=True, max_items=None):
    import soundfile as sf
    vf, c, _spec = _load_vf(ckpt, use_ema, device="cuda" if torch.cuda.is_available() else "cpu")
    dev = next(vf.parameters()).device
    eng, synth, idx_map = make_engine()
    B = len(target_wavs) if max_items is None else min(max_items, len(target_wavs))
    cond = torch.from_numpy(np.asarray(clap_emb[:B], np.float32)).to(dev)
    cand = sample_best_of_n(vf, cond, _spec.dim, N,
                            steps or c.sample_steps, s or c.cfg_strength, dev).cpu().numpy()
    best, mean = [], []
    for b in range(B):
        tgt, sr_t = sf.read(target_wavs[b])
        if tgt.ndim > 1: tgt = tgt.mean(axis=1)
        if sr_t != RENDER_SR:
            print(f"  [WARN] {target_wavs[b]}: sr={sr_t} ≠ {RENDER_SR}: comparison is biased.")
        ds_ = []
        for k in range(N):
            pr = enforce_audio_logic(_spec.decode(cand[k, b]))
            ds_.append(mss_distance(render_preset(eng, synth, idx_map, pr), tgt.astype(np.float32)))
        best.append(min(ds_)); mean.append(float(np.mean(ds_)))
        if (b + 1) % 25 == 0: print(f"  ... {b+1}/{B}")
    print(f"\n── Audio-space (N={N}, B={B}) " + "─" * 30)
    print(f"  MSS best-of-{N} : {np.mean(best):.4f} ± {np.std(best):.4f}")
    print(f"  MSS mean       : {np.mean(mean):.4f}")
    print("─" * 58)
    return {"mss_best": best, "mss_mean": mean}

## 8d. Inference sweep (guidance scale, steps, t*)

In [ ]:
def sweep_cfg_steps(ckpt, split="val", n=500,
                    s_grid=(1.0, 1.5, 2.0, 3.0, 4.0), step_grid=(20, 40)):
    ds = PresetDataset(cfg, spec, split)
    ids = ds.ids[:n]; clap = ds.clap[ids]
    rows = []
    for s in s_grid:
        for st in step_grid:
            preds = predict_from_audio(ckpt, clap, steps=st, s=s)
            cont, acc = [], []
            for k, i in enumerate(ids):
                tgt = spec.decode(ds.x[i])
                for p, _ in spec.layout:
                    e = abs(preds[k][p.name] - tgt[p.name])
                    if p.kind == "cont": cont.append(e)
                    elif p.kind == "cat": acc.append(1.0 if preds[k][p.name] == tgt[p.name] else 0.0)
            rows.append((s, st, float(np.mean(cont)), float(np.mean(acc))))
            print(f"  s={s:3.1f} steps={st:3d} → MAE={rows[-1][2]:.4f}  cat_acc={rows[-1][3]:.3f}")
    return rows

def sweep_t_star(ckpt, clap_target, init_params, t_grid=(0.3, 0.5, 0.7, 0.9), steps=40, s=2.0):
    out = {}
    for ts in t_grid:
        preds = predict_edit(ckpt, clap_target, init_params, t_star=ts, steps=steps, s=s)
        moved = [np.mean([abs(p[k] - float(init_params.get(k, 0.0))) for k in p]) for p in preds]
        out[ts] = {"preds": preds, "moved_L1": float(np.mean(moved))}
    print("\n  t*    | L1 shift")
    for ts, r in out.items(): print(f"  {ts:.2f}  | {r['moved_L1']:.4f}")
    return out

## 9. Text adapter

In [ ]:
class TextAdapter(nn.Module):
    def __init__(self, dim=512, h=1024):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, h), nn.GELU(), nn.Linear(h, dim))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)
    def forward(self, z): return z + self.net(z)

def train_text_adapter(z_text: np.ndarray, z_audio: np.ndarray,
                       epochs=50, bs=512, lr=1e-4, lam=0.5, device="cuda"):
    zt = torch.from_numpy(z_text.astype(np.float32))
    za = torch.from_numpy(z_audio.astype(np.float32))
    ad = TextAdapter().to(device)
    opt = torch.optim.AdamW(ad.parameters(), lr=lr)
    n = len(zt)
    for ep in range(epochs):
        perm, tot = torch.randperm(n), 0.0
        for i in range(0, n, bs):
            b = perm[i:i+bs]
            xt, xa = zt[b].to(device), za[b].to(device)
            zh = ad(xt)
            loss = (1 - F.cosine_similarity(zh, xa, -1).mean()) + lam * F.mse_loss(zh, xa)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(b)
        if (ep + 1) % 10 == 0: print(f"  epoch {ep+1:3d}  loss={tot/n:.4f}")
    return ad

## 10. Regression baseline

In [ ]:
class RegressionNet(nn.Module):
    def __init__(self, spec, d_token, n_heads, n_layers, clap_dim, d_cond):
        super().__init__()
        self.spec = spec
        n_params = len(spec.params)
        self.p_emb = nn.Embedding(n_params, d_token)
        self.clap_mlp = nn.Sequential(nn.Linear(clap_dim, d_cond), nn.SiLU(),
                                      nn.Linear(d_cond, d_cond))
        self.blocks = nn.ModuleList([DiTBlock(d_token, n_heads, d_cond) for _ in range(n_layers)])
        self.final  = FinalLayer(d_token, d_cond)
        self.out_p  = nn.ModuleList([nn.Linear(d_token, len(p)) for p, _ in spec.layout])

    def forward(self, cond):
        B = cond.shape[0]
        z = self.clap_mlp(cond)
        toks = self.p_emb(torch.arange(len(self.spec.params), device=cond.device))
        toks = toks.unsqueeze(0).expand(B, -1, -1)
        for blk in self.blocks:
            toks = blk(toks, z)
        toks = self.final(toks, z)
        return torch.cat([self.out_p[i](toks[:, i]) for i in range(len(self.spec.layout))], dim=1)

def train_regression_baseline(cfg, max_steps=None):
    _param_names = load_param_names(cfg.locked_params_json)
    _spec = ParamSpec(_param_names)
    ds = PresetDataset(cfg, _spec, "train")
    w = ds.sampler_weights(cfg.handcrafted_weight)
    samp = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double),
                                 num_samples=len(ds), replacement=True)
    dl = DataLoader(ds, batch_size=cfg.batch_size, sampler=samp, num_workers=2,
                    pin_memory=True, drop_last=True, collate_fn=plain_collate)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net = RegressionNet(_spec, cfg.d_token, cfg.n_heads, cfg.n_layers,
                        cfg.clap_dim, cfg.d_cond).to(device)
    print(f"[OK] RegressionNet: {sum(p.numel() for p in net.parameters())/1e6:.2f}M parameters")
    ema = EMA(net, cfg.ema_decay)

    steps = max_steps or cfg.max_steps
    opt = torch.optim.AdamW(net.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    def lr_lambda(s_):
        if s_ < cfg.warmup_steps: return (s_ + 1) / max(1, cfg.warmup_steps)
        p = (s_ - cfg.warmup_steps) / max(1, steps - cfg.warmup_steps)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    val_ds = PresetDataset(cfg, _spec, "val")
    g = torch.Generator().manual_seed(cfg.seed)
    nv = min(cfg.val_batch, len(val_ds))
    vi = torch.randperm(len(val_ds), generator=g)[:nv]
    v_clap = torch.stack([val_ds[i][0] for i in vi]).to(device)
    v_x1   = torch.stack([val_ds[i][1] for i in vi]).to(device)

    step, best = 0, float("inf")
    net.train()
    while step < steps:
        for clap, x1, _ in dl:
            clap, x1 = clap.to(device), x1.to(device)
            opt.zero_grad(set_to_none=True)
            loss = (net(clap) - x1).square().mean()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), cfg.grad_clip)
            opt.step(); sched.step(); ema.update(net); step += 1
            if step % 500 == 0:
                print(f"  step {step:>6d}/{steps}  mse={loss.item():.4f}  lr={sched.get_last_lr()[0]:.2e}")
            if step % cfg.val_every == 0:
                net.eval()
                with torch.no_grad():
                    vl = (net(v_clap) - v_x1).square().mean().item()
                net.train()
                if vl < best: best = vl
                print(f"  [VAL] step {step:>6d}  val_mse={vl:.4f}  (best={best:.4f})")
            if step >= steps: break

    out = os.path.join(cfg.out_dir, "regression_baseline.pt")
    torch.save({"net": net.state_dict(), "ema": ema.cpu_state(),
                "cfg": cfg.__dict__, "param_names": _param_names}, out)
    print(f"[OK] Baseline saved: {out}  (best val_mse={best:.4f})")
    return out

def predict_regression(ckpt_path, clap_audio, use_ema=True, device="cuda"):
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    c = CFG(**{k: v for k, v in ck["cfg"].items() if k in CFG.__dataclass_fields__})
    _spec = ParamSpec(ck["param_names"])
    net = RegressionNet(_spec, c.d_token, c.n_heads, c.n_layers, c.clap_dim, c.d_cond).to(device)
    net.load_state_dict(ck["ema"] if (use_ema and "ema" in ck) else ck["net"]); net.eval()
    cond = torch.from_numpy(np.asarray(clap_audio, np.float32)).to(device)
    with torch.no_grad():
        x = net(cond).cpu().numpy()
    return [enforce_audio_logic(_spec.decode(x[i])) for i in range(len(x))]